# Atividade Extra — IA Embarcada e Modelos Compactos

## Classificação termo-higrométrica com dados de agricultura inteligente

**Objetivo:** treinar e compactar um modelo simples usando temperatura e umidade de uma base agrícola e, depois, executar a inferência em um ESP32-S3.

Na etapa embarcada, a proposta é usar um **DHT22** no Wokwi.

**Fluxo:** dataset → exploração → comparação de arquiteturas → Float32 → INT8 → inferência → ESP32-S3.

> As classes ambientais criadas aqui são didáticas e não representam diagnóstico agronômico.

## 1. Contextualização do problema

Em IA embarcada, o modelo precisa apresentar bom desempenho sem consumir recursos em excesso. Por isso, além da acurácia, serão observados número de parâmetros e tamanho do arquivo TensorFlow Lite.

A aplicação recebe **temperatura** e **umidade do ar** e classifica a condição termo-higrométrica. Essas duas grandezas podem ser lidas pelo DHT22.

## 2. Dataset

Dataset: **Smart Farming Sensor Data for Yield Prediction**, de Atharva Soundankar, disponível no Kaggle.

Fonte: https://www.kaggle.com/datasets/atharvasoundankar/smart-farming-sensor-data-for-yield-prediction

A base possui **500 registros e 22 colunas** de agricultura inteligente. Para esta aplicação serão usadas somente:

- `temperature_C`
- `humidity_%`

Como a base já é pequena, os registros serão mantidos sem redução.

## 3. Instalação e bibliotecas

In [ ]:
!pip -q install kagglehub

: 

In [ ]:
# Bibliotecas usadas na exploração, treinamento e exportação do modelo
import os
import glob
import json
import zipfile
import shutil
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

import tensorflow as tf

# Semente fixa para deixar os experimentos mais reproduzíveis
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Paleta visual usada ao longo do notebook
# A, B e C mantêm sempre as mesmas cores nos comparativos.
CORES_MODELOS = {
    "A": "#1f77b4",  # azul
    "B": "#ff7f0e",  # laranja
    "C": "#2ca02c"   # verde
}

# As quatro classes ambientais também recebem cores fixas.
CORES_CLASSES = {
    "AMENO": "#1f77b4",
    "INTERMEDIARIO": "#ff7f0e",
    "QUENTE_SECO": "#2ca02c",
    "QUENTE_UMIDO": "#d62728"
}

COR_FLOAT32 = "#1f77b4"
COR_INT8 = "#ff7f0e"

# Ciclo padrão para gráficos que não informarem cor manualmente
plt.rcParams["axes.prop_cycle"] = plt.cycler(
    color=["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]
)

print("TensorFlow:", tf.__version__)
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)


## 4. Download do dataset

In [ ]:
import kagglehub

DATASET_ID = "atharvasoundankar/smart-farming-sensor-data-for-yield-prediction"

try:
    dataset_path = kagglehub.dataset_download(DATASET_ID)
    csv_files = glob.glob(os.path.join(dataset_path, "**", "*.csv"), recursive=True)
    if not csv_files:
        raise FileNotFoundError("Nenhum CSV encontrado.")
    csv_path = csv_files[0]
    print("Dataset:", dataset_path)
    print("CSV:", csv_path)
except Exception as erro:
    print("Download automático não concluído:", erro)
    from google.colab import files
    uploaded = files.upload()
    csv_path = list(uploaded.keys())[0]

## 5. Leitura e conferência

In [ ]:
# Faz a leitura do CSV e uma conferência inicial da base
df = pd.read_csv(csv_path)

print("Dimensão:", df.shape)
print("\nColunas:")
print(df.columns.tolist())

display(df.head())


In [ ]:
print("Valores ausentes:")
display(df.isnull().sum().to_frame("ausentes"))

print("Duplicados:", df.duplicated().sum())

## 6. Seleção das entradas

O modelo usará somente temperatura e umidade, pois ambas podem ser obtidas pelo DHT22.

In [ ]:
# Mantém apenas as duas entradas que serão usadas no ESP32-S3
FEATURES = ["temperature_C", "humidity_%"]

faltando = [c for c in FEATURES if c not in df.columns]
if faltando:
    raise ValueError(f"Colunas não encontradas: {faltando}")

dados = df[FEATURES].copy()

# Garante que as duas colunas sejam numéricas e remove registros inválidos
dados["temperature_C"] = pd.to_numeric(dados["temperature_C"], errors="coerce")
dados["humidity_%"] = pd.to_numeric(dados["humidity_%"], errors="coerce")
dados = dados.dropna().drop_duplicates().reset_index(drop=True)

print("Registros usados:", len(dados))
display(dados.head())


## 7. Criação da classe de saída

Para transformar a base em uma aplicação de classificação embarcada, foi criada a variável `condicao_ambiental`:

- **AMENO:** temperatura abaixo de 22 °C;
- **QUENTE_SECO:** temperatura ≥ 28 °C e umidade < 65%;
- **QUENTE_UMIDO:** temperatura ≥ 28 °C e umidade ≥ 65%;
- **INTERMEDIARIO:** demais situações.

Essa regra serve apenas para o experimento da disciplina.

In [ ]:
# Regra simples criada para transformar temperatura e umidade em classes
def classificar_condicao(linha):
    t = linha["temperature_C"]
    h = linha["humidity_%"]

    if t < 22:
        return "AMENO"
    if t >= 28 and h < 65:
        return "QUENTE_SECO"
    if t >= 28 and h >= 65:
        return "QUENTE_UMIDO"
    return "INTERMEDIARIO"

dados["condicao_ambiental"] = dados.apply(classificar_condicao, axis=1)

display(dados.head(10))


## 8. Exploração dos dados

In [ ]:
display(dados[FEATURES].describe().round(2))

dist = dados["condicao_ambiental"].value_counts()
perc = dados["condicao_ambiental"].value_counts(normalize=True).mul(100).round(2)

tabela_classes = pd.DataFrame({
    "quantidade": dist,
    "percentual_%": perc
})

display(tabela_classes)

In [ ]:
# Distribuição das classes usando uma cor fixa para cada rótulo
contagem_classes = dados["condicao_ambiental"].value_counts()
cores = [CORES_CLASSES.get(classe, "#7f7f7f") for classe in contagem_classes.index]

plt.figure(figsize=(8, 4))
contagem_classes.plot(kind="bar", color=cores)
plt.title("Distribuição das classes")
plt.xlabel("Classe")
plt.ylabel("Quantidade")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


In [ ]:
# Temperatura e umidade são mostradas com cores diferentes para facilitar a leitura
plt.figure(figsize=(8, 4))
plt.hist(dados["temperature_C"], bins=20, color="#1f77b4")
plt.title("Distribuição da temperatura")
plt.xlabel("Temperatura (°C)")
plt.ylabel("Frequência")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
plt.hist(dados["humidity_%"], bins=20, color="#ff7f0e")
plt.title("Distribuição da umidade")
plt.xlabel("Umidade (%)")
plt.ylabel("Frequência")
plt.tight_layout()
plt.show()


In [ ]:
classes_plot = sorted(dados["condicao_ambiental"].unique())

plt.figure(figsize=(8, 5))

for classe in classes_plot:
    parte = dados[dados["condicao_ambiental"] == classe]
    plt.scatter(
        parte["temperature_C"],
        parte["humidity_%"],
        s=30,
        alpha=0.65,
        label=classe,
        color=CORES_CLASSES.get(classe, "#7f7f7f")
    )

plt.title("Temperatura × umidade por classe")
plt.xlabel("Temperatura (°C)")
plt.ylabel("Umidade (%)")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
estatisticas = (
    dados.groupby("condicao_ambiental")[FEATURES]
    .agg(["mean", "std", "min", "max"])
    .round(2)
)

display(estatisticas)

## 9. Preparação dos dados

In [ ]:
# Converte os nomes das classes em rótulos numéricos
label_encoder = LabelEncoder()

X = dados[FEATURES].astype(np.float32).values
y = label_encoder.fit_transform(dados["condicao_ambiental"])

class_names = label_encoder.classes_.tolist()
num_classes = len(class_names)

for i, nome in enumerate(class_names):
    print(i, "->", nome)

# Separa treino e teste preservando a proporção das classes
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=SEED,
    stratify=y
)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)


## 10. Normalização

In [ ]:
# A normalização é ajustada apenas com os dados de treino
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
X_test_scaled = scaler.transform(X_test).astype(np.float32)

# Estes valores também serão levados para o código embarcado
feature_means = scaler.mean_.astype(np.float32)
feature_scales = scaler.scale_.astype(np.float32)

for nome, media, desvio in zip(FEATURES, feature_means, feature_scales):
    print(f"{nome}: média={media:.6f} | desvio={desvio:.6f}")


# 11. Comparação de arquiteturas

| Arquitetura | Estrutura |
|---|---|
| A | 2 → Dense(4) → saída |
| B | 2 → Dense(8) → saída |
| C | 2 → Dense(8) → Dense(4) → saída |

Serão comparados acurácia, loss, parâmetros e tamanho do `.tflite`.

In [ ]:
def criar_modelo(nome):
    if nome == "A":
        camadas = [
            tf.keras.layers.Input(shape=(2,)),
            tf.keras.layers.Dense(4, activation="relu"),
            tf.keras.layers.Dense(num_classes, activation="softmax")
        ]
    elif nome == "B":
        camadas = [
            tf.keras.layers.Input(shape=(2,)),
            tf.keras.layers.Dense(8, activation="relu"),
            tf.keras.layers.Dense(num_classes, activation="softmax")
        ]
    elif nome == "C":
        camadas = [
            tf.keras.layers.Input(shape=(2,)),
            tf.keras.layers.Dense(8, activation="relu"),
            tf.keras.layers.Dense(4, activation="relu"),
            tf.keras.layers.Dense(num_classes, activation="softmax")
        ]
    else:
        raise ValueError("Arquitetura inválida")

    modelo = tf.keras.Sequential(camadas)

    modelo.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return modelo

## 12. Treinamento

In [ ]:
# Treina as três redes nas mesmas condições para permitir uma comparação justa
modelos = {}
historicos = {}
resultados = []

for nome in ["A", "B", "C"]:
    print(f"\n===== ARQUITETURA {nome} =====")

    tf.keras.backend.clear_session()
    tf.random.set_seed(SEED)

    modelo = criar_modelo(nome)

    # Interrompe o treinamento quando a validação para de melhorar
    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=15,
        restore_best_weights=True
    )

    historico = modelo.fit(
        X_train_scaled,
        y_train,
        validation_split=0.20,
        epochs=150,
        batch_size=16,
        callbacks=[early_stop],
        verbose=0
    )

    loss, acc = modelo.evaluate(X_test_scaled, y_test, verbose=0)

    # Converte temporariamente para TFLite para comparar também o tamanho
    converter_tmp = tf.lite.TFLiteConverter.from_keras_model(modelo)
    tflite_tmp = converter_tmp.convert()

    resultados.append({
        "arquitetura": nome,
        "parametros": modelo.count_params(),
        "epocas": len(historico.history["loss"]),
        "acuracia": acc,
        "loss": loss,
        "tamanho_tflite_kb": len(tflite_tmp) / 1024
    })

    modelos[nome] = modelo
    historicos[nome] = historico

    print("Parâmetros:", modelo.count_params())
    print(f"Acurácia: {acc*100:.2f}%")
    print(f"Loss: {loss:.4f}")
    print(f"TFLite: {len(tflite_tmp)/1024:.2f} KB")


## 13. Comparativo

In [ ]:
comparacao = pd.DataFrame(resultados)

comparacao["acuracia_%"] = (comparacao["acuracia"] * 100).round(2)
comparacao["loss"] = comparacao["loss"].round(4)
comparacao["tamanho_tflite_kb"] = comparacao["tamanho_tflite_kb"].round(2)

tabela_comparacao = comparacao[
    ["arquitetura", "parametros", "epocas", "acuracia_%", "loss", "tamanho_tflite_kb"]
]

display(tabela_comparacao)

In [ ]:
plt.figure(figsize=(7, 4))

cores_arquiteturas = [
    CORES_MODELOS[nome]
    for nome in tabela_comparacao["arquitetura"]
]

plt.bar(
    tabela_comparacao["arquitetura"],
    tabela_comparacao["acuracia_%"],
    color=cores_arquiteturas
)

plt.title("Acurácia por arquitetura")
plt.xlabel("Arquitetura")
plt.ylabel("Acurácia (%)")

for i, v in enumerate(tabela_comparacao["acuracia_%"]):
    plt.text(i, v + 0.4, f"{v:.2f}%", ha="center")

plt.ylim(max(0, tabela_comparacao["acuracia_%"].min() - 5), 100)
plt.tight_layout()
plt.show()


In [ ]:
cores_arquiteturas = [
    CORES_MODELOS[nome]
    for nome in tabela_comparacao["arquitetura"]
]

plt.figure(figsize=(7, 4))
plt.bar(
    tabela_comparacao["arquitetura"],
    tabela_comparacao["parametros"],
    color=cores_arquiteturas
)
plt.title("Quantidade de parâmetros")
plt.xlabel("Arquitetura")
plt.ylabel("Parâmetros")
for i, v in enumerate(tabela_comparacao["parametros"]):
    plt.text(i, v, str(v), ha="center", va="bottom")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
plt.bar(
    tabela_comparacao["arquitetura"],
    tabela_comparacao["tamanho_tflite_kb"],
    color=cores_arquiteturas
)
plt.title("Tamanho do modelo TFLite")
plt.xlabel("Arquitetura")
plt.ylabel("Tamanho (KB)")
for i, v in enumerate(tabela_comparacao["tamanho_tflite_kb"]):
    plt.text(i, v, f"{v:.2f}", ha="center", va="bottom")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))

for nome in ["A", "B", "C"]:
    plt.plot(
        historicos[nome].history["val_accuracy"],
        label=f"Arquitetura {nome}",
        color=CORES_MODELOS[nome]
    )

plt.title("Acurácia de validação")
plt.xlabel("Época")
plt.ylabel("Acurácia")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


### Interpretação do comparativo

A arquitetura **B** apresentou o melhor resultado no teste, com **99% de acurácia**. A arquitetura **C** ficou muito próxima, com **98%**, enquanto a arquitetura **A** chegou a **86%**.

O resultado mostra que aumentar a capacidade da rede A para a B trouxe um ganho importante. Já a camada extra da arquitetura C não trouxe melhora de acurácia e ainda deixou o modelo maior. Para uma aplicação embarcada, a arquitetura B ficou com o melhor equilíbrio entre desempenho e simplicidade.


## 14. Seleção do modelo compacto

Modelos até **1 ponto percentual** abaixo da melhor acurácia serão considerados equivalentes. Entre eles, será escolhido o que tiver menos parâmetros.

In [ ]:
# Considera equivalentes os modelos que ficaram até 1 ponto percentual do melhor
melhor_acc = comparacao["acuracia"].max()

candidatos = comparacao[
    comparacao["acuracia"] >= (melhor_acc - 0.01)
].copy()

# Entre os equivalentes, prioriza menos parâmetros e menor arquivo
candidatos = candidatos.sort_values(
    by=["parametros", "tamanho_tflite_kb", "acuracia"],
    ascending=[True, True, False]
)

melhor_nome = candidatos.iloc[0]["arquitetura"]

model = modelos[melhor_nome]

test_loss, test_accuracy = model.evaluate(
    X_test_scaled,
    y_test,
    verbose=0
)

print("Arquitetura selecionada:", melhor_nome)
print("Parâmetros:", model.count_params())
print(f"Acurácia: {test_accuracy*100:.2f}%")
print(f"Loss: {test_loss:.4f}")


### Interpretação da escolha

A arquitetura **B** foi selecionada com **60 parâmetros** e **99% de acurácia**. Como a arquitetura C ficou dentro da margem de 1 ponto percentual, mas possui mais parâmetros, não havia vantagem prática em escolher a rede mais complexa.

A escolha prioriza um modelo pequeno, mas sem abrir mão do desempenho obtido no conjunto de teste.


## 15. Avaliação detalhada

In [ ]:
y_prob = model.predict(X_test_scaled, verbose=0)
y_pred = np.argmax(y_prob, axis=1)

print(
    classification_report(
        y_test,
        y_pred,
        target_names=class_names,
        digits=4,
        zero_division=0
    )
)

cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names
)

fig, ax = plt.subplots(figsize=(7, 6))
disp.plot(ax=ax, cmap="YlGnBu", values_format="d")
plt.title(f"Matriz de confusão — Arquitetura {melhor_nome}")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

### Leitura da avaliação

A acurácia de **99%** indica que o modelo teve desempenho alto no conjunto de teste. O relatório de classificação ajuda a verificar esse comportamento em cada classe, enquanto a matriz de confusão mostra onde ocorreram os erros.

Como as classes foram criadas a partir de limites de temperatura e umidade, o problema possui regiões bem definidas. Por isso, o resultado deve ser entendido dentro do caráter didático deste experimento, e não como uma classificação agronômica real.


# 16. Conversão Float32

In [ ]:
converter_float = tf.lite.TFLiteConverter.from_keras_model(model)
modelo_float32 = converter_float.convert()

FLOAT_FILE = "smart_farming_float32.tflite"

with open(FLOAT_FILE, "wb") as f:
    f.write(modelo_float32)

float_kb = len(modelo_float32) / 1024

print(f"Tamanho Float32: {float_kb:.2f} KB")

## 17. Inferência Float32

In [ ]:
interpreter_float = tf.lite.Interpreter(model_path=FLOAT_FILE)
interpreter_float.allocate_tensors()

input_float = interpreter_float.get_input_details()[0]
output_float = interpreter_float.get_output_details()[0]

amostra = X_test_scaled[0:1].astype(np.float32)

interpreter_float.set_tensor(input_float["index"], amostra)
interpreter_float.invoke()

saida_float = interpreter_float.get_tensor(output_float["index"])[0]

classe_float = int(np.argmax(saida_float))
classe_real = int(y_test[0])

print("Classe real:", class_names[classe_real])
print("Classe Float32:", class_names[classe_float])
print("Saída:", saida_float)

# 18. Quantização INT8

In [ ]:
# Usa amostras reais de treino para calibrar a quantização
def representative_dataset():
    limite = min(150, len(X_train_scaled))

    for i in range(limite):
        yield [X_train_scaled[i:i+1].astype(np.float32)]

converter_int8 = tf.lite.TFLiteConverter.from_keras_model(model)

converter_int8.optimizations = [tf.lite.Optimize.DEFAULT]
converter_int8.representative_dataset = representative_dataset

# Força operações, entrada e saída em INT8
converter_int8.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS_INT8
]

converter_int8.inference_input_type = tf.int8
converter_int8.inference_output_type = tf.int8

modelo_int8 = converter_int8.convert()

INT8_FILE = "smart_farming_int8.tflite"

with open(INT8_FILE, "wb") as f:
    f.write(modelo_int8)

int8_kb = len(modelo_int8) / 1024

print(f"Tamanho INT8: {int8_kb:.2f} KB")


## 19. Inferência INT8

In [ ]:
interpreter_int8 = tf.lite.Interpreter(model_path=INT8_FILE)
interpreter_int8.allocate_tensors()

input_int8 = interpreter_int8.get_input_details()[0]
output_int8 = interpreter_int8.get_output_details()[0]

print("Entrada:", input_int8["dtype"], input_int8["quantization"])
print("Saída:", output_int8["dtype"], output_int8["quantization"])

input_scale, input_zero_point = input_int8["quantization"]
output_scale, output_zero_point = output_int8["quantization"]

def inferir_int8(amostra_float):
    q = np.round(amostra_float / input_scale + input_zero_point)
    q = np.clip(q, -128, 127).astype(np.int8)

    interpreter_int8.set_tensor(input_int8["index"], q)
    interpreter_int8.invoke()

    saida_q = interpreter_int8.get_tensor(output_int8["index"])[0]

    saida = (
        saida_q.astype(np.float32) - output_zero_point
    ) * output_scale

    classe = int(np.argmax(saida))

    return classe, saida_q, saida

classe_int8, saida_q, saida_dequant = inferir_int8(
    X_test_scaled[0:1]
)

print("Classe real:", class_names[classe_real])
print("Classe Float32:", class_names[classe_float])
print("Classe INT8:", class_names[classe_int8])

## 20. Avaliação do INT8

In [ ]:
pred_int8 = []

for i in range(len(X_test_scaled)):
    classe, _, _ = inferir_int8(X_test_scaled[i:i+1])
    pred_int8.append(classe)

pred_int8 = np.array(pred_int8)

accuracy_int8 = (pred_int8 == y_test).mean()

print(f"Acurácia Float32: {test_accuracy*100:.2f}%")
print(f"Acurácia INT8: {accuracy_int8*100:.2f}%")

### 20.1 Avaliação detalhada do modelo INT8

A acurácia geral mostra se a quantização preservou o desempenho, mas também é importante verificar cada classe separadamente. Por isso, o modelo INT8 é avaliado novamente com relatório de classificação e matriz de confusão.


In [ ]:
# Avaliação por classe do modelo quantizado
print("RELATÓRIO DE CLASSIFICAÇÃO — INT8\n")

print(
    classification_report(
        y_test,
        pred_int8,
        target_names=class_names,
        digits=4,
        zero_division=0
    )
)

cm_int8 = confusion_matrix(y_test, pred_int8)

disp_int8 = ConfusionMatrixDisplay(
    confusion_matrix=cm_int8,
    display_labels=class_names
)

fig, ax = plt.subplots(figsize=(7, 6))
disp_int8.plot(ax=ax, cmap="YlOrBr", values_format="d")
plt.title("Matriz de confusão — Modelo INT8")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


### Interpretação do INT8

O modelo quantizado manteve **99% de acurácia**, o mesmo resultado obtido antes da quantização. Isso indica que a conversão para INT8 não provocou perda relevante de capacidade de classificação neste conjunto de teste.

A matriz de confusão e as métricas por classe servem para confirmar que o bom resultado não ficou restrito apenas à acurácia geral.


### 20.2 Concordância Float32 × INT8

Além de comparar a acurácia, vale verificar quantas vezes os dois modelos chegaram exatamente à mesma decisão para as mesmas amostras.


In [ ]:
# Compara diretamente as previsões feitas pelas duas versões do modelo
concordancia = (y_pred == pred_int8).mean() * 100
delta_acc = (accuracy_int8 - test_accuracy) * 100

print(f"Concordância Float32 × INT8: {concordancia:.2f}%")
print(f"Variação de acurácia após quantização: {delta_acc:+.2f} p.p.")
print(f"Tamanho Float32: {float_kb:.2f} KB")
print(f"Tamanho INT8: {int8_kb:.2f} KB")

if concordancia >= 99:
    print("\nInterpretação: as duas versões tomaram praticamente as mesmas decisões no conjunto de teste.")
elif concordancia >= 95:
    print("\nInterpretação: houve alta concordância entre as versões, com poucas mudanças de decisão após a quantização.")
else:
    print("\nInterpretação: a quantização alterou uma parte perceptível das decisões e merece análise mais detalhada.")


# 21. Comparação Float32 × INT8

In [ ]:
comparacao_quantizacao = pd.DataFrame({
    "modelo": ["Float32", "INT8"],
    "acuracia_%": [
        round(test_accuracy * 100, 2),
        round(accuracy_int8 * 100, 2)
    ],
    "tamanho_kb": [
        round(float_kb, 2),
        round(int8_kb, 2)
    ],
    "entrada": ["float32", "int8"],
    "saida": ["float32", "int8"]
})

display(comparacao_quantizacao)

In [ ]:
cores_quantizacao = [COR_FLOAT32, COR_INT8]

plt.figure(figsize=(7, 4))
plt.bar(
    comparacao_quantizacao["modelo"],
    comparacao_quantizacao["tamanho_kb"],
    color=cores_quantizacao
)
plt.title("Tamanho: Float32 × INT8")
plt.xlabel("Modelo")
plt.ylabel("Tamanho (KB)")
for i, v in enumerate(comparacao_quantizacao["tamanho_kb"]):
    plt.text(i, v, f"{v:.2f} KB", ha="center", va="bottom")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
plt.bar(
    comparacao_quantizacao["modelo"],
    comparacao_quantizacao["acuracia_%"],
    color=cores_quantizacao
)
plt.title("Acurácia: Float32 × INT8")
plt.xlabel("Modelo")
plt.ylabel("Acurácia (%)")
for i, v in enumerate(comparacao_quantizacao["acuracia_%"]):
    plt.text(i, v + 0.4, f"{v:.2f}%", ha="center")
plt.ylim(max(0, comparacao_quantizacao["acuracia_%"].min() - 5), 100)
plt.tight_layout()
plt.show()


### Interpretação da quantização

As duas versões atingiram **99% de acurácia**, então a quantização preservou o desempenho observado no teste.

O arquivo Float32 ficou com aproximadamente **1,92 KB**, enquanto o INT8 ficou com cerca de **2,27 KB**. Neste caso, a versão INT8 não ficou menor. Isso acontece porque a rede é extremamente pequena e o overhead do próprio formato TFLite passa a ter peso maior no tamanho final do arquivo.

Mesmo assim, o INT8 continua sendo adequado para a etapa embarcada por trabalhar com entrada, saída e operações inteiras, que é justamente o formato preparado para a inferência no ESP32-S3.


# 22. Arquivos para o ESP32-S3

In [ ]:
def bytes_para_cpp(data):
    linhas = []

    for i in range(0, len(data), 12):
        bloco = data[i:i+12]
        linha = ", ".join(f"0x{b:02x}" for b in bloco)
        linhas.append("  " + linha + ",")

    return "\n".join(linhas)

array_cpp = bytes_para_cpp(modelo_int8)

cc_lines = [
    '#include "model_data.h"',
    "",
    "alignas(16) const unsigned char g_model[] = {",
    array_cpp,
    "};",
    "",
    f"const unsigned int g_model_len = {len(modelo_int8)};"
]

h_lines = [
    "#ifndef MODEL_DATA_H_",
    "#define MODEL_DATA_H_",
    "",
    "extern const unsigned char g_model[];",
    "extern const unsigned int g_model_len;",
    "",
    "#endif"
]

with open("model_data.cc", "w", encoding="utf-8") as f:
    f.write("\n".join(cc_lines))

with open("model_data.h", "w", encoding="utf-8") as f:
    f.write("\n".join(h_lines))

print("model_data.cc e model_data.h criados")

## 23. Configuração para normalização e quantização

In [ ]:
def cpp_float(v):
    return f"{float(v):.9f}f"

classes_cpp = ", ".join(f'"{c}"' for c in class_names)

config_lines = [
    "#ifndef MODEL_CONFIG_H_",
    "#define MODEL_CONFIG_H_",
    "",
    f"static constexpr float FEATURE_MEAN[2] = {{{cpp_float(feature_means[0])}, {cpp_float(feature_means[1])}}};",
    f"static constexpr float FEATURE_STD[2] = {{{cpp_float(feature_scales[0])}, {cpp_float(feature_scales[1])}}};",
    f"static constexpr float INPUT_SCALE = {cpp_float(input_scale)};",
    f"static constexpr int INPUT_ZERO_POINT = {int(input_zero_point)};",
    f"static constexpr float OUTPUT_SCALE = {cpp_float(output_scale)};",
    f"static constexpr int OUTPUT_ZERO_POINT = {int(output_zero_point)};",
    f"static constexpr int NUM_CLASSES = {num_classes};",
    f"static const char* CLASS_NAMES[NUM_CLASSES] = {{{classes_cpp}}};",
    "",
    "#endif"
]

with open("model_config.h", "w", encoding="utf-8") as f:
    f.write("\n".join(config_lines))

print("\n".join(config_lines))

## 24. Metadados e ZIP

In [ ]:
metadata = {
    "dataset": DATASET_ID,
    "records_used": int(len(dados)),
    "features": FEATURES,
    "target": "condicao_ambiental",
    "classes": class_names,
    "selected_architecture": melhor_nome,
    "parameters": int(model.count_params()),
    "float32_accuracy": float(test_accuracy),
    "int8_accuracy": float(accuracy_int8),
    "float32_size_kb": float(float_kb),
    "int8_size_kb": float(int8_kb),
    "feature_mean": [float(x) for x in feature_means],
    "feature_std": [float(x) for x in feature_scales],
    "input_scale": float(input_scale),
    "input_zero_point": int(input_zero_point),
    "output_scale": float(output_scale),
    "output_zero_point": int(output_zero_point)
}

with open("model_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print(json.dumps(metadata, indent=2, ensure_ascii=False))

In [ ]:
modelo_dir = Path("modelo")
modelo_dir.mkdir(exist_ok=True)

arquivos = [
    "smart_farming_float32.tflite",
    "smart_farming_int8.tflite",
    "model_data.cc",
    "model_data.h",
    "model_config.h",
    "model_metadata.json"
]

for arquivo in arquivos:
    shutil.copy2(arquivo, modelo_dir / arquivo)

ZIP_FILE = "modelo_smart_farming_esp32.zip"

with zipfile.ZipFile(ZIP_FILE, "w", zipfile.ZIP_DEFLATED) as zipf:
    for arquivo in arquivos:
        zipf.write(modelo_dir / arquivo, arcname=f"modelo/{arquivo}")

print("ZIP criado:", ZIP_FILE)

## 25. Download

In [ ]:
from google.colab import files
files.download("modelo_smart_farming_esp32.zip")

# Conclusão

O experimento comparou três arquiteturas compactas para classificar condições termo-higrométricas a partir de temperatura e umidade. A arquitetura **B** apresentou o melhor equilíbrio para a proposta, com **60 parâmetros** e **99% de acurácia**, superando a arquitetura A (86%) e ficando à frente da arquitetura C (98%).

Depois da conversão para TensorFlow Lite, foram avaliadas as versões Float32 e INT8. A quantização manteve a acurácia em **99%**, mostrando que, neste experimento, foi possível usar representação inteira sem perda de desempenho no conjunto de teste.

Em tamanho de arquivo, o Float32 ficou em aproximadamente **1,92 KB** e o INT8 em **2,27 KB**. Como a rede é muito pequena, o overhead do formato TFLite acaba tendo influência maior e, por isso, o arquivo INT8 não ficou menor neste caso.

A versão INT8 foi exportada com os parâmetros de normalização e quantização necessários para a etapa embarcada, utilizando **ESP32-S3 + DHT22 + TensorFlow Lite Micro**. As classes criadas são apenas didáticas e servem para demonstrar o fluxo completo de uma aplicação de IA embarcada.
